In [1]:
import math
import os
import sys
sys.path.append(os.path.abspath('.')) # to run files that are away
os.environ["WANDB_SILENT"] = "true"  # Suppress WandB logs

libraries = ["torch", "numpy", "polars"]
modules   = {lib: sys.modules.get(lib) for lib in libraries}

if not modules["torch"]:
    import torch
if not modules["numpy"]:
    import numpy as np
if not modules["polars"]:
    import polars as pl

import pandas as pd
import gc
import catboost as cb
import lightgbm as lgb
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from load_and_rename_files import LogFilesProcessor, WaferFilesProcessor
from asm_utils import count_missing_values_in_df

main_folder   = "../ASM_data"
NUM_WAFERS    = 4
step_col_name = "step_id"
COMMON_ID_COLS= ['Process Time', '#Run']
COMMON_ID_COLS_MOD  = [c for c in COMMON_ID_COLS if c != "#Run"] + ["marathon_run"]

parquet_folder_name = "wafer_parquet_files"
dict_of_wafer_files = {'file1': {'path': f"{main_folder}/2. marathon0/Wafer performance/Spatial property after step 4.csv", 'marathon': 0},
                       'file2': {'path': f"{main_folder}/3. marathon1/Wafer performance/Spatial property.csv", 'marathon': 1}}

dict_of_log_files = {'file1': {'path': f"{main_folder}/2. marathon0/logs/Step1.csv", 'step': 1, 'marathon': 0},
                     'file2': {'path': f"{main_folder}/2. marathon0/logs/Step2.csv", 'step': 2, 'marathon': 0},
                     'file3': {'path': f"{main_folder}/2. marathon0/logs/Step3.csv", 'step': 3, 'marathon': 0},
                     'file4': {'path': f"{main_folder}/2. marathon0/logs/Step4.csv", 'step': 4, 'marathon': 0},
                     'file5': {'path': f"{main_folder}/3. marathon1/logs/Step1.csv", 'step': 1, 'marathon': 1},
                     'file6': {'path': f"{main_folder}/3. marathon1/logs/Step2.csv", 'step': 2, 'marathon': 1},
                     'file7': {'path': f"{main_folder}/3. marathon1/logs/Step3.csv", 'step': 3, 'marathon': 1},
                     'file8': {'path': f"{main_folder}/3. marathon1/logs/Step4.csv", 'step': 4, 'marathon': 1},}

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# file with nulls:
# df = pl.read_parquet(f'../ASM_data/{parquet_folder_name}/wafer_1_log.parquet')

# df = pl.read_csv(f'../ASM_data/2. marathon0/logs/Step1.csv', ignore_errors=True)
# df = pl.read_csv(f'../ASM_data/2. marathon0/logs/Step1.csv', null_values=["", "NA", "null"], ignore_errors=True, separator=",")

pdf = pd.read_csv(f'../ASM_data/2. marathon0/logs/Step1.csv', decimal='.')
df  = pl.from_pandas(pdf)

count_missing_values_in_df(df)

count = 0
for i, row in enumerate(df.iter_rows(named=True)):
    for col, val in row.items():
        if val is None:
            print(f"Null at row {i}, column '{col}'")
            count += 1
            if count == 5:
                break
    if count == 5:
        break


# columns with NaNs: 0
# rows with NaNs: 0
Total NaNs: 0
# columns with nulls: 0
# rows with nulls: 0
Total nulls: 0


In [3]:
# master_wafer_df, wafer_df_dict, y_df_dict, radius_wide_dict=load_and_preprocess_wafer_data(dict_of_wafer_files, main_folder, save=False)
# unique_marathon_runs_list = list(master_wafer_df["marathon_run"].unique())
# log_df = load_and_process_log_files(dict_of_log_files, log_processor, unique_marathon_runs_list, step_col_name, main_folder, save=False)
# split_and_save_log_df_by_wafer(log_df, NUM_WAFERS, main_folder, log_processor, overwrite = True)

# master_
# df.head()


In [18]:

def load_and_preprocess_wafer_csv_data(dict_of_wafer_files, main_folder: str, save: bool = False) -> tuple:
    """Merge wafer files, split by RC, and create y and radius dataframes"""
    processor = WaferFilesProcessor()
    master_wafer_df = processor.load_and_merge_wafer_files(dict_of_wafer_files)
    if save:
        master_wafer_df.write_parquet(f"{main_folder}/{parquet_folder_name}/master_wafer_file.parquet")
    wafer_df_dict = processor.split_wafer_df_by_rc(master_wafer_df)

    y_dict, radius_dict, radius_wide_dict = {}, {}, {}
    for i, df in wafer_df_dict.items():
        y_dict[i], radius_dict[i] = processor.split_1_wafer_df_to_y_and_radius_df(df)
        r_df = radius_dict[i].sort("marathon_run").with_columns(
            pl.arange(0, pl.len()).over("marathon_run").alias("radius_idx"))
        radius_wide_dict[i] = r_df.pivot(
            values="Radius (mm)", index="marathon_run", on="radius_idx", aggregate_function="first").sort("marathon_run")
    return master_wafer_df, wafer_df_dict, y_dict, radius_wide_dict

def load_and_process_log_csv_files(dict_of_log_files, log_processor: LogFilesProcessor, unique_marathon_runs_list: list, step_col_name: str,main_folder: str,save: bool = False):
    """Read all stepfile CSVs, concat, then optionally save to parquet"""
    df_list = []

    for entry in dict_of_log_files.values():
        df = log_processor.read_csv_and_rename_cols(entry['path'])
        df = log_processor.add_marathon_and_step_columns(df, entry['marathon'], entry['step'], step_col_name)
        df = log_processor.remove_runs_not_found_in_wafer_df(df, unique_marathon_runs_list)
        df = log_processor.insert_step_cols_after_run(df, step_col_name)
        df = log_processor.cast_int_and_float_to_float64(df)
        df = log_processor.drop_single_value_cols(df, step_col_name)
        df = log_processor.append_step_suffix_to_cols(df, step_col_name, entry['step'])
        df_list.append(df)

    master_log_df = pl.concat(df_list, how="diagonal")
    master_log_df2 = log_processor.reorder_cols(master_log_df, step_col_name)
    log_df        = master_log_df2.rename({c: c.strip().lower() for c in master_log_df2.columns})
    return df, df_list, master_log_df, log_df

log_processor = LogFilesProcessor(COMMON_ID_COLS_MOD, COMMON_ID_COLS)

master_wafer_df, wafer_df_dict, y_df_dict, radius_wide_dict = load_and_preprocess_wafer_csv_data(dict_of_wafer_files, main_folder, save=False)
unique_marathon_runs_list = list(master_wafer_df["marathon_run"].unique())
df_out, df_list, master_log_df, log_df = load_and_process_log_csv_files(dict_of_log_files, log_processor, unique_marathon_runs_list, step_col_name, main_folder, save=False)


In [23]:
# wafer_log_df = pl.read_parquet(f"{main_folder}/{parquet_folder_name}/wafer_{1}_log.parquet")
# wafer_log_df.head()

# df_out.head(10)
# count_missing_values_in_df(df_out)

# master_log_df.head()
log_df.head()

process time,marathon_run,step_id,#run,common signal_5_step1,common signal_6_step1,common signal_7_step1,common signal_34_step1,common signal_35_step1,common signal_36_step1,common signal_42_step1,common signal_46_step1,common signal_47_step1,common signal_48_step1,common signal_49_step1,common signal_50_step1,common signal_51_step1,common signal_52_step1,common signal_53_step1,common signal_54_step1,common signal_55_step1,common signal_56_step1,common signal_58_step1,common signal_59_step1,common signal_60_step1,common signal_62_step1,common signal_63_step1,common signal_66_step1,common signal_67_step1,common signal_69_step1,common signal_70_step1,common signal_71_step1,common signal_72_step1,common signal_74_step1,common signal_75_step1,common signal_76_step1,common signal_80_step1,…,common signal_96_step4,common signal_97_step4,common signal_98_step4,common signal_99_step4,common signal_100_step4,common signal_102_step4,common signal_103_step4,common signal_104_step4,common signal_105_step4,common signal_107_step4,common signal_108_step4,common signal_109_step4,common signal_110_step4,common signal_111_step4,common signal_112_step4,common signal_113_step4,common signal_114_step4,common signal_115_step4,common signal_118_step4,common signal_119_step4,common signal_120_step4,common signal_121_step4,common signal_122_step4,common signal_123_step4,common signal_153_step4,common signal_154_step4,common signal_155_step4,common signal_156_step4,common signal_157_step4,rc1 signal_5_step4,rc1 signal_6_step4,rc2 signal_5_step4,rc2 signal_6_step4,rc3 signal_5_step4,rc3 signal_6_step4,rc4 signal_5_step4,rc4 signal_6_step4
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2024-08-02T14:30:42.976""","""0_1""",1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,40.2,0.1,27.1,0.0,53.6,0.1,0.0,67.1,0.1,-0.1,15.8,0.3,6.3,0.3,2000.3,67.1,11.9,0.8,15.8,0.0,0.7,0.7,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""2024-08-02T14:30:43.026""","""0_1""",1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,40.1,0.1,27.1,0.0,53.6,0.1,0.0,67.1,0.1,-0.1,15.8,0.3,6.3,0.3,2000.0,67.1,11.9,0.8,15.8,0.0,0.7,0.7,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""2024-08-02T14:30:43.076""","""0_1""",1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,40.1,0.1,27.1,0.0,53.6,0.1,0.0,67.1,0.1,-0.1,15.8,0.3,6.3,0.3,1999.9,67.1,11.9,0.8,15.8,0.0,0.7,0.7,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""2024-08-02T14:30:43.126""","""0_1""",1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,40.1,0.1,27.1,0.0,53.6,0.1,0.0,67.1,0.1,-0.1,15.8,0.3,6.3,0.3,2000.0,67.1,11.9,0.8,15.8,0.0,0.7,0.7,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""2024-08-02T14:30:43.176""","""0_1""",1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,40.1,0.1,27.1,0.0,53.6,0.1,0.0,67.1,0.1,-0.1,15.8,0.3,6.3,0.3,1999.9,67.1,11.9,0.8,15.8,0.0,0.7,0.7,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [6]:
def read_csv_and_rename_cols(file_path: str) -> pl.DataFrame:
    """Read CSV into Polars DataFrame and title-case column names after stripping spaces"""
    df = pl.read_csv(file_path, ignore_errors=True)
    df = df.rename({c: c.strip().title() for c in df.columns})
    return df

# count_missing_values_in_df(dict_of_log_files[1])


In [7]:
y_pred_cat.shape
y_pred_unscaled = y_scaler.inverse_transform(y_pred_cat)

type(y_pred_cat)

NameError: name 'y_pred_cat' is not defined

In [ ]:
y_pred_unscaled = y_scaler.inverse_transform(y_pred_cat)

plt.scatter(range(len(y_full_pd.iloc[1].values)), y_full_pd.iloc[1].values,
            label="True", facecolors='none', edgecolors='blue', s=8)
plt.scatter(range(len(y_pred_unscaled[1])), y_pred_unscaled[1],
            label="Predicted", facecolors='none', edgecolors='orange', s=8)
# plt.plot(y_full_pd.iloc[1].values, label="True")
# plt.plot(y_pred_unscaled[1], label="Predicted")
plt.legend()
plt.title("y_predicted vs y_actual")
plt.xlabel("Site ID (coordinate)")
plt.ylabel("Spatial property")
plt.show()


In [ ]:
# hyperparam search

from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'estimator__num_leaves': [20, 31, 40, 50],
    'estimator__max_depth': [-1, 5, 10, 20],
    'estimator__min_data_in_leaf': [10, 20, 30],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__n_estimators': [100, 500, 1000]}

# model = MultiOutputRegressor(xgb.XGBRegressor(objective='reg:squarederror', verbosity=0))
model = MultiOutputRegressor(LGBMRegressor(objective='regression', verbosity=-1))

search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=20,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV RMSE:", (-search.best_score_)**0.5)



##### Spatial data (M)

In [ ]:
def plot_wafer_property(df, property_col, title):
    plt.figure(figsize=(9, 5))
    for rc_value, group in df.group_by("RC"):
        x = group["#Run"].to_list()
        y = group[property_col].to_list()
        plt.scatter(x, y, label=f'RC {rc_value}', s=20)
    plt.xlabel("#Run")
    plt.ylabel(property_col)
    plt.title(title)
    plt.legend()
    plt.show()

plot_wafer_property(wafer_df, "Wafer property summary 1", "Wafer Property 1")
plot_wafer_property(wafer_df, "Wafer property summary 2", "Wafer Property 2")


##### Import timeseries data (S)

In [ ]:
"""Load data and make parquet files out of it"""

should_we_save_parquet_files = False

def _save_df_as_parquet_file(df: pl.dataframe, saving_location: str):
    df.write_parquet(saving_location)

def remove_unchanging_cols_from_df_and_save(df: pl.dataframe, col_name: str, should_we_save_parquet_files: bool) -> None:
    for run_id in df[col_name].unique().to_list():
        df_per_run    = df.filter(pl.col(col_name) == run_id)
        constant_cols = [col for col in df_per_run.columns
                     if df_per_run[col].dtype in [pl.Int64, pl.Float64] and
                        #  df_per_run.select(pl.col(col).filter(pl.col(col) != 0)).height == 0]
                         df_per_run.select(pl.col(col).n_unique()).item() == 1]
        df_per_run_filtered = df_per_run.drop(constant_cols)
        saving_location = f"{parquet_subfolder}/run_{run_id}.parquet"
        if should_we_save_parquet_files:
            _save_df_as_parquet_file(df_per_run_filtered, saving_location)

remove_unchanging_cols_from_df_and_save(log_df, "#Run", should_we_save_parquet_files)

# =============
# before making funcrtion:

# if should_we_save_parquet_files:
#     for run_id in log_df["#Run"].unique().to_list():
#         df_per_run = log_df.filter(pl.col("#Run") == run_id)
#         zero_cols  = [col for col in df_per_run.columns
#                      if df_per_run[col].dtype in [pl.Int64, pl.Float64] and
#                         #  df_per_run.select(pl.col(col).filter(pl.col(col) != 0)).height == 0]
#                          df_per_run.select(pl.col(col).n_unique()).item() == 1]
#         df_per_run_filtered = df_per_run.drop(zero_cols)
#         df_per_run_filtered.write_parquet(f"{parquet_subfolder}/run_{run_id}.parquet")


##### Timeseries data (S)

In [ ]:
run_number   = 15
parquet_file = f"./ASM_data/3. marathon1/Logs/split_by_run/run_{run_number}.parquet"
df           = pl.read_parquet(parquet_file)
df_pd        = df.to_pandas()
df_numeric   = df.select(pl.col(pl.NUMERIC_DTYPES))
X_np         = df_numeric.to_numpy()
X_scaled     = StandardScaler().fit_transform(X_np)

num_cols_to_plot = len(df_pd.columns)
num_rows         = math.ceil(math.sqrt(num_cols_to_plot))
num_cols_grid    = math.ceil(num_cols_to_plot / num_rows)

axes = df_pd.plot(subplots=True, figsize=(14, 12), layout=(num_rows, num_cols_grid), sharex=True, legend=False)

if isinstance(axes, np.ndarray):
    axes_flat = axes.flatten()
else:
    axes_flat = [axes]

column_names = df_pd.columns.tolist()

for i, ax in enumerate(axes_flat):
    if i < num_cols_to_plot: # Only set title for actual plots
        ax.set_title(column_names[i], fontsize='xx-small')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])

for i in range(num_cols_to_plot, len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.tight_layout()
plt.suptitle(f'Run #{run_number} {log_file}', fontsize='large', y=1.02) # Adjust y to prevent overlap
plt.show()

In [ ]:
data_dimensionality = DimensionalityEstimator.estimate_dataset_dimensionality(df)
print(f"Recommended latent layer size: {data_dimensionality:.1f}")


In [ ]:
from sklearn.model_selection import train_test_split


X_train_np, X_test_np = train_test_split(X_scaled, test_size=0.2, random_state=42)
X_train               = torch.tensor(X_train_np, dtype=torch.float32)
X_test                = torch.tensor(X_test_np, dtype=torch.float32)
